In [1]:
import pandas as pd

df = pd.read_csv("messy_sales.csv")

print("Shape:", df.shape)
display(df.head(10))
print("\nData Types:\n" , df.dtypes)


Shape: (300, 6)


,order_id,date,product,price,qty,zip
0,1254,4/6/26,mug,11.36,4,60614
1,1057,5/30/26,charger,14.79,3,30303
2,1150,5/6/26,mug,5.50,1,10001
3,1066,5/30/26,notebook,NaN,1,98101
4,1114,4/6/26,webcam,45.59,4,90405
5,1280,4/22/26,pen set,NaN,4,30303
6,1204,4/18/26,charger,8.41,2,2134
7,1249,5/15/26,keyboard,75.04,4,90405
8,1037,5/19/26,mug,9.31,2,60614
9,1177,5/13/26,mug,10.11,3,90210



Data Types:
 order_id      int64
date            str
product         str
price       float64
qty           int64
zip           int64
dtype: object


Observations:
Several rows such as rows 3 and 5 are missing price values (NaN).
There are zipcodes stored as integers, from what I notice is that the zip was read as as int64, and leading zeros were dropped. For example: row 6 is displaying 2134 instead of a five digit zip code.
Also I see dates stored as text, the date column has a string (str) data type instead of a datetime type, which will prevent chronological sorting or date math. 

In [2]:
df.isna().sum()

order_id     0
date         0
product      0
price       12
qty          0
zip          0
dtype: int64

In [3]:
fill_value = df["price"].median()
print("Median fill value:", fill_value)

df["price"] = df["price"].fillna(fill_value)
df.isna().sum()

Median fill value: 37.53


order_id    0
date        0
product     0
price       0
qty         0
zip         0
dtype: int64

In [4]:
print("Duplictaes found:", df.duplicated().sum())
df = df.drop_duplicates()
print("New shape:", df.shape)

Duplictaes found: 8
New shape: (292, 6)


In [5]:
df["zip"] = df["zip"].astype(str).str.zfill(5)
df.head(10)

,order_id,date,product,price,qty,zip
0,1254,4/6/26,mug,11.36,4,60614
1,1057,5/30/26,charger,14.79,3,30303
2,1150,5/6/26,mug,5.50,1,10001
3,1066,5/30/26,notebook,37.53,1,98101
4,1114,4/6/26,webcam,45.59,4,90405
5,1280,4/22/26,pen set,37.53,4,30303
6,1204,4/18/26,charger,8.41,2,02134
7,1249,5/15/26,keyboard,75.04,4,90405
8,1037,5/19/26,mug,9.31,2,60614
9,1177,5/13/26,mug,10.11,3,90210


In [6]:
df["date"] = pd.to_datetime(df["date"], format="mixed")
print(df.dtypes)

order_id             int64
date        datetime64[us]
product                str
price              float64
qty                  int64
zip                    str
dtype: object


In [7]:
df[df["qty"] < 0]

,order_id,date,product,price,qty,zip
202,1140,2026-04-18,webcam,38.69,-5,98101
262,1233,2026-05-09,desk lamp,22.64,-4,10001
297,1025,2026-04-27,keyboard,47.30,-2,02116


In [8]:
df = df[df["qty"] >= 0]
print("New shape:", df.shape)

New shape: (289, 6)


I removed the three rows with the negative quantities which are, order_id 1140, 1233, and 1025. Because you can't buy a negative number of items and there is no note explaining if they were refunds or typos. So by removing those leaves 289 clean rows so unverified numbers do not throw off our revenue totals.

In [9]:
df.to_csv("sales_clean.csv", index=False)

I have started off with 300 rows of sales data and then I fixed five main problems to prepare the file for analysis. First I filled 12 missing prices using the median price of 37.53 so that the really expensive items would not pull the average up. I then removed 8 duplicated rows which brought the total down to 292 rows. After, I changed the zip codes from numbers to 5 character text so leading zeros came back. And I converted the dates into one consistent date time format. Finally I then deleted the 3 rows with negative quantities because there were no proof they were valid returns, leaving 289 clean rows. 

I think if a data analyst made the wrong choice here, such as leaving duplicate orders in or keeping unverified negative quantities the company could miscalculate the total revenue and then make poor decisions about the inventory and the restocking.